# Ticha HTR dataset pipeline

From the Ticha website to a train / validation / test line dataset with augmentation.

| Step | What | Output (under `data/`) |
|---|---|---|
| 1 | Scrape the manuscript list, transcriptions and page images | `manuscripts.csv`, `texts.csv`, `images.csv`, `images/` |
| 2 | Summary of the downloaded data | `reports/summary_*.csv` |
| 3 | Discard documents without a usable transcription (and their images) | `texts_transcribed.csv`, `images_transcribed.csv`, `excluded/` |
| 4 | Segment pages into lines (kraken) and pair them with the transcription | `paired/` (`auto/`, `review/`, `manifest.csv`, `lines.csv`) |
| 5 | Report: auto-paired pages vs. pages needing review | `reports/review_pages.csv` |
| 6 | Apply the manual review (`review/decisions.txt`) and choose the final pages | `reviewed/`, `reports/final_pages.csv` |
| 7 | Split documents into train / validation / test (80 / 10 / 10) | `splits/splits.csv` |
| 8 | Cut lines and pair them with their text, per split | `splits/<split>/lines/`, `splits/<split>/auto/` |
| 9 | Data augmentation, kept apart from the splits | `augmented/<set>/` |
| 10 | Final dataset metrics | `reports/final_metrics.csv` |

**Before running**: create the `ticha` conda environment and select the **Python (ticha)** kernel (see [`ENVIRONMENT.md`](../ENVIRONMENT.md)):

```bash
conda env create -f environment.yml
conda activate ticha
python -m ipykernel install --user --name ticha --display-name "Python (ticha)"
```

The notebook imports the code straight from `src/` in this repo; the `ticha` package does not need to be installed. Steps 6-10 call the functions of [`scripts/build_dataset.py`](../scripts/build_dataset.py), so the notebook and the script build the same dataset, including the documents left out in [`scripts/dataset_exclusions.py`](../scripts/dataset_exclusions.py).

- Every step saves its output and is **skipped when the output already exists** (set `OVERWRITE = True` to redo steps 4-10). Downloads are reused unless `REDOWNLOAD = True`; an interrupted download resumes where it stopped, page by page.
- For a first test set `MAX_DOCS = 3` in the configuration cell, then delete `data/` and run everything with `MAX_DOCS = None`.
- The full download is slow: ~200 documents, several pages each, tile by tile, with rate limiting (expect hours). Step 4 is also slow (kraken segmentation + recognition per page): `WORKERS = 2` processes 2 documents at a time (~1.6x faster segmentation on an M2 Air with 8 GB; 3 is the most that fits in memory). With an NVIDIA GPU, use `DEVICE = "cuda:0"`.
- Splits are made **by document**, so pages and lines of one document never end up in two subsets.

## 0. Configuration

In [ ]:
import json
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Use the code in this repo (src/), not an installed ticha package
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

import pandas as pd
from PIL import Image

from ticha import TichaImageScraper, TichaScraper, TichaTextScraper
from ticha.core.review import apply_review
from ticha.core.line_pairing import (
    DEFAULT_MODEL,
    DEFAULT_STATUSES,
    KrakenRecognizer,
    KrakenSegmenter,
    LinePairer,
    split_transcription_pages,
    transcription_lines,
)

import build_dataset as build  # steps 6-10, shared with scripts/build_dataset.py
from dataset_exclusions import DATASET_VERSION, EXCLUDED_DOCS

import ticha
assert Path(ticha.__file__).is_relative_to(ROOT / "src"), f"ticha imported from {ticha.__file__}, expected {ROOT / 'src'}"

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

In [ ]:
# --- Paths ------------------------------------------------------------------
DATA = Path(os.environ.get("TICHA_DATA", ROOT / "data"))   # set TICHA_DATA to keep the data elsewhere

MANUSCRIPTS_CSV = DATA / "manuscripts.csv"          # step 1: list of documents
TEXTS_CSV = DATA / "texts.csv"                      # step 1: transcriptions + metadata
IMAGES_DIR = DATA / "images"                        # step 1: page images, one folder per document
IMAGES_CSV = DATA / "images.csv"                    # step 1: index of downloaded images
KEPT_TEXTS_CSV = DATA / "texts_transcribed.csv"     # step 3
KEPT_IMAGES_CSV = DATA / "images_transcribed.csv"   # step 3
EXCLUDED_DIR = DATA / "excluded"                    # step 3: images of discarded documents
PAIRED_DIR = DATA / "paired"                        # step 4: output of the line pairer
REVIEWED_DIR = DATA / "reviewed"                    # step 6: PAGE XML corrected by hand
SPLITS_DIR = DATA / "splits"                        # steps 7-8: original lines only
AUGMENTED_DIR = DATA / "augmented"                  # step 9: augmented copies, kept apart from splits/
REPORTS_DIR = DATA / "reports"

for d in (DATA, REPORTS_DIR, REVIEWED_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Run control ------------------------------------------------------------
MAX_DOCS = None       # e.g. 3 for a quick end-to-end test; None = every document
OVERWRITE = False     # True = redo steps 4-10 even if their output exists (never re-downloads)
REDOWNLOAD = False    # True = download the list, texts and images again (step 1). Leave False:
                      # finished documents and pages already on disk are always reused

# --- Step 1: download -------------------------------------------------------
RATE_LIMIT = 2.0      # seconds between documents (be gentle with the server)
TILE_WORKERS = 4      # image tiles downloaded at the same time (Ticha's own viewer also loads several)
IMAGE_FORMAT = "png"
MAX_DIMENSION = None  # e.g. 3000 to cap the longest side; None = full resolution

# --- Step 3: what counts as transcribed -------------------------------------
KEEP_STATUSES = DEFAULT_STATUSES   # ("Completed and Checked", "Completed but not Checked"); None = any status

# --- Step 4: segmentation and pairing ---------------------------------------
DEVICE = "cpu"            # keep "cpu" on Apple Silicon (kraken stalls on "mps"); "cuda:0" with an NVIDIA GPU
WORKERS = 2               # documents processed in parallel; ~2 GB RAM each (with 8 GB of RAM: 2, at most 3)
# Segmentation uses kraken's default baseline model (blla.mlmodel), fixed by kraken==7.1.1 in environment.yml.
# Pairing reads each line with an HTR model and cuts the transcription into the spans that match the readings.
GENERIC_MODEL = DEFAULT_MODEL   # McCATMuS (Zenodo DOI): generic model for the first pass
# Where the Ticha fine-tuned model (used for the final pairing) comes from:
#   "download": the published model; reproduces the published dataset
#   "train":    fine-tune McCATMuS here on the auto/ pages of a first pass (about 7 hours on an M2 CPU)
#   "generic":  no fine-tuning, pair with McCATMuS only (more pages end up in review)
FINETUNED_SOURCE = "download"
FINETUNED_REPO = "maclenn77/ticha-htr-mccatmus-ft"   # Hugging Face model repository
FINETUNED_FILE = "ticha_ft_e06.safetensors"
FINETUNED_DIR = DATA / "ticha_ft"
FINETUNE_EPOCHS = 7       # epochs 0-6; the published model is the checkpoint of epoch 6 (validation accuracy 86.1%)
GENERIC_PAIRED_DIR = DATA / "paired_mccatmus"       # first pass, only with FINETUNED_SOURCE = "train"
# Confidence of a pair: CER up to CER_THRESHOLD, or up to MAX_CER when its text fits the reading at
# least MIN_MARGIN better than the neighbouring lines' texts. A page goes to auto/ when at most
# REVIEW_TOLERANCE of its pairs are doubtful (those lines are kept there without text).
CER_THRESHOLD = 0.5
MAX_CER = 0.75
MIN_MARGIN = 0.15
REVIEW_TOLERANCE = 0.2

# --- Step 6: review and final pages -----------------------------------------
# Line-by-line review of the review/ pages, kept in the repository as a decisions file
# ("doc page line keep|drop|fix [text]"). It reproduces the review of the published dataset.
# None: review by hand instead (e.g. in eScriptorium) and put the corrected PAGE XML in reviewed/.
REVIEW_DECISIONS = ROOT / "review" / "decisions.txt"
# False: pages flagged for review are used only once a reviewed PAGE XML is in reviewed/.
# True:  unreviewed pages are also used, keeping only their confident lines (conf >= MIN_CONF).
INCLUDE_UNREVIEWED = False
MIN_CONF = 0.5

# --- Step 7: split ----------------------------------------------------------
SPLIT_RATIOS = build.SPLIT_RATIOS   # 80 / 10 / 10 of the lines within each language
SEED = 42

# --- Step 9: augmentation ---------------------------------------------------
# Augmented copies go to AUGMENTED_DIR, never into splits/, and stay in the subset
# of their original line. Train gets degrade + variety copies; validation and test get
# separate degraded-only copies (no variety, no geometry) to measure how much worse the
# model reads low-quality scans. For other options (imitating your own scans, whole
# pages) use `ticha augment`.
COPIES = 4                      # augmented copies per train line image
DEGRADE_RATIO = 0.5             # share of copies imitating Ticha's low-quality untranscribed scans
TEST_DEGRADED_COPIES = 1        # degraded copies per test line (0 = no degraded test set)
VALIDATION_DEGRADED_COPIES = 1  # the same for validation (not in the published dataset)


def done(path: Path) -> bool:
    # A step is skipped when its output exists, unless OVERWRITE is set.
    return path.exists() and not OVERWRITE


def downloaded(path: Path) -> bool:
    # Downloads are reused unless REDOWNLOAD is set (OVERWRITE does not apply).
    return path.exists() and not REDOWNLOAD


print(f"Data directory: {DATA}")

## 1. Scrape and download all the manuscripts

Three passes over the site:

1. **Manuscript list** (Selenium, needs Chrome): one row per document with its metadata (`language`, `trptn_status`, `doc_type`, `year`, ...).
2. **Transcriptions** of every document (plain HTTP).
3. **Page images**: the PDF links on Ticha are broken, so each page is rebuilt from its Deep Zoom tiles into `images/<ticha_id>/page_NNN.png`.

Downloads are done **once**:

- Passes 2 and 3 save a checkpoint after every document. Re-running the cell (or the whole notebook) skips finished documents, and within an unfinished document skips the pages already on disk; only failed documents are retried.
- `OVERWRITE` never re-downloads. Only `REDOWNLOAD = True` does.
- Tiles are fetched `TILE_WORKERS` at a time (4 by default); a full page is ~165 tiles.

In [ ]:
# 1a. Manuscript list
if downloaded(MANUSCRIPTS_CSV):
    manuscripts = pd.read_csv(MANUSCRIPTS_CSV)
    print(f"Loaded {MANUSCRIPTS_CSV.name}")
else:
    manuscripts = TichaScraper(rate_limit=RATE_LIMIT).scrape_manuscripts()  # closes its browser itself
    if manuscripts.empty:
        raise RuntimeError("No manuscripts scraped; check the website / Chrome driver")
    manuscripts.to_csv(MANUSCRIPTS_CSV, index=False)

manuscripts = manuscripts[manuscripts["document_link"].notna() & (manuscripts["document_link"] != "")]
if MAX_DOCS:
    manuscripts = manuscripts.head(MAX_DOCS)
print(f"{len(manuscripts)} documents to process")
manuscripts.head()

In [ ]:
# 1b. Transcriptions and document metadata (resumable)
def scrape_texts(manuscripts: pd.DataFrame) -> pd.DataFrame:
    previous = pd.read_csv(TEXTS_CSV) if downloaded(TEXTS_CSV) else pd.DataFrame()
    finished = set(previous["url"]) if "url" in previous else set()
    rows = previous.to_dict("records")
    todo = manuscripts[~manuscripts["document_link"].isin(finished)]
    scraper = TichaTextScraper(rate_limit=RATE_LIMIT)
    try:
        for i, (_, row) in enumerate(todo.iterrows(), 1):
            print(f"[{i}/{len(todo)}] {row['document_link']}")
            rows.append({**row.to_dict(), **scraper.scrape_document(row["document_link"])})
            pd.DataFrame(rows).to_csv(TEXTS_CSV, index=False)  # checkpoint
    finally:
        scraper.close()
    return pd.DataFrame(rows)


texts = scrape_texts(manuscripts)
texts = texts[texts["url"].isin(manuscripts["document_link"])]
failed = texts[texts["error"].notna()] if "error" in texts else texts.iloc[:0]
print(f"Texts scraped: {len(texts)} documents, {len(failed)} with errors")
failed[["url", "error"]] if len(failed) else None

In [ ]:
# 1c. Page images (resumable)
def download_images(manuscripts: pd.DataFrame) -> pd.DataFrame:
    previous = pd.read_csv(IMAGES_CSV) if downloaded(IMAGES_CSV) else pd.DataFrame()
    if "missing_folios" in previous:  # retry documents that failed or are missing pages
        previous = previous[previous["error"].isna() & previous["missing_folios"].isna()]
    else:  # written before missing pages were tracked: re-check every document once
        previous = previous.iloc[0:0]  # (pages already on disk are not downloaded again)
    finished = set(previous["url"]) if "url" in previous else set()
    rows = previous.to_dict("records")
    todo = manuscripts[~manuscripts["document_link"].isin(finished)]
    scraper = TichaImageScraper(rate_limit=RATE_LIMIT, tile_workers=TILE_WORKERS)
    for i, (_, row) in enumerate(todo.iterrows(), 1):
        print(f"[{i}/{len(todo)}] {row['document_link']}")
        result = scraper.download_document_images(
            row["document_link"],
            IMAGES_DIR,
            image_format=IMAGE_FORMAT,
            max_dimension=MAX_DIMENSION,
            try_pdf=False,  # Ticha's PDF links return 500; pages are rebuilt from tiles
            skip_existing=not REDOWNLOAD,  # pages already on disk are not downloaded again
        )
        result["image_count"] = len(result["image_paths"])
        result["image_paths"] = ";".join(result["image_paths"])
        result["image_folios"] = ";".join(result["image_folios"])
        result["missing_folios"] = ";".join(result["missing_folios"]) or None
        rows.append({**row.to_dict(), **result})
        pd.DataFrame(rows).to_csv(IMAGES_CSV, index=False)  # checkpoint
    return pd.DataFrame(rows)


images = download_images(manuscripts)
images = images[images["url"].isin(manuscripts["document_link"])]
failed = images[images["error"].notna() | images["missing_folios"].notna()]
print(f"Images: {int(images['image_count'].sum())} pages from {(images['image_count'] > 0).sum()} documents; "
      f"{images['error'].notna().sum()} documents with errors, "
      f"{(images['error'].isna() & images['missing_folios'].notna()).sum()} with missing pages")
failed[["ticha_id", "url", "image_count", "missing_folios", "error"]] if len(failed) else None

## 2. Summary of the downloaded data

Transcribed lines are counted the same way the pairer will split them (one line per transcription line; column layouts separated by 4+ spaces count as separate lines; folio markers like `(8r)` are not lines).

In [ ]:
def count_lines(transcription) -> int:
    if not isinstance(transcription, str) or not transcription.strip():
        return 0
    return sum(len(transcription_lines(text)) for _, text in split_transcription_pages(transcription))


def count_chars(transcription) -> int:
    if not isinstance(transcription, str):
        return 0
    return sum(len(line) for _, text in split_transcription_pages(transcription) for line in transcription_lines(text))


text_cols = ["url", "transcription"] + [c for c in ("error",) if c in texts]
docs = (
    manuscripts
    .merge(texts[text_cols].rename(columns={"url": "document_link", "error": "text_error"}), on="document_link", how="left")
    .merge(images[["url", "image_count", "missing_folios", "error"]].rename(columns={"url": "document_link", "error": "image_error"}),
           on="document_link", how="left")
)
docs["image_count"] = docs["image_count"].fillna(0).astype(int)
docs["has_transcription"] = docs["transcription"].fillna("").str.strip().ne("")
docs["transcribed_lines"] = docs["transcription"].map(count_lines)
docs["transcribed_chars"] = docs["transcription"].map(count_chars)
for col in ("language", "trptn_status", "doc_type", "legibility"):
    docs[col] = docs[col].fillna("(unknown)")

totals = {
    "documents listed": len(docs),
    "documents with images downloaded": int((docs["image_count"] > 0).sum()),
    "pages downloaded": int(docs["image_count"].sum()),
    "documents with download errors": int(docs["image_error"].notna().sum()),
    "documents with missing pages": int(docs["missing_folios"].notna().sum()),
    "documents with a transcription": int(docs["has_transcription"].sum()),
    "transcribed lines": int(docs["transcribed_lines"].sum()),
    "transcribed characters": int(docs["transcribed_chars"].sum()),
    "pages of transcribed documents": int(docs.loc[docs["has_transcription"], "image_count"].sum()),
}
totals = pd.Series(totals, name="value").to_frame()
totals.to_csv(REPORTS_DIR / "summary_totals.csv")
totals

In [ ]:
def breakdown(by: str) -> pd.DataFrame:
    g = docs.groupby(by)
    table = pd.DataFrame({
        "documents": g.size(),
        "transcribed docs": g["has_transcription"].sum(),
        "pages": g["image_count"].sum(),
        "transcribed lines": g["transcribed_lines"].sum(),
        "transcribed chars": g["transcribed_chars"].sum(),
    })
    for col in ("documents", "pages", "transcribed lines"):
        table[f"% {col}"] = (100 * table[col] / max(table[col].sum(), 1)).round(1)
    table = table.sort_values("documents", ascending=False)
    table.loc["TOTAL"] = table.sum(numeric_only=True)
    table = table.astype({c: int for c in table.columns if not c.startswith("%")})
    table.to_csv(REPORTS_DIR / f"summary_by_{by}.csv")
    return table


print("By language (Spanish / Zapotec)")
display(breakdown("language"))
print("By transcription status")
display(breakdown("trptn_status"))
print("By document type")
display(breakdown("doc_type"))
print("By legibility")
display(breakdown("legibility"))

years = pd.to_numeric(docs["year"], errors="coerce")
print(f"Years: {int(years.min()) if years.notna().any() else '?'} - {int(years.max()) if years.notna().any() else '?'}; "
      f"documents per century:")
(years // 100 * 100).value_counts().sort_index().rename("documents").to_frame()

## 3. Discard documents without a transcription (and their images)

A document is kept when it has a transcription **and** its status is in `KEEP_STATUSES` (by default "Completed and Checked" and "Completed but not Checked"; "In Progress" transcriptions are partial, so their lines would not match the page).

Images of discarded documents are **moved** (not deleted) to `excluded/images/`, so nothing is lost if you change your mind.

In [ ]:
# Unknown statuses are kept if the document has a transcription (same rule as the pairer).
if KEEP_STATUSES:
    status_ok = docs["trptn_status"].isin(KEEP_STATUSES) | (docs["trptn_status"] == "(unknown)")
else:
    status_ok = pd.Series(True, index=docs.index)
docs["keep"] = docs["has_transcription"] & status_ok & (docs["image_count"] > 0)
docs["discard_reason"] = None
docs.loc[docs["image_count"] == 0, "discard_reason"] = "no images"
docs.loc[~status_ok, "discard_reason"] = "status: " + docs["trptn_status"]
docs.loc[~docs["has_transcription"], "discard_reason"] = "no transcription"

kept_urls = set(docs.loc[docs["keep"], "document_link"])
texts[texts["url"].isin(kept_urls)].to_csv(KEPT_TEXTS_CSV, index=False)
images[images["url"].isin(kept_urls)].to_csv(KEPT_IMAGES_CSV, index=False)

# Move the image folders of discarded documents out of images/
moved = 0
for paths in images.loc[~images["url"].isin(kept_urls), "image_paths"].dropna():
    for folder in {Path(p).parent for p in str(paths).split(";") if p}:
        if folder.exists() and folder.parent == IMAGES_DIR:
            target = EXCLUDED_DIR / "images" / folder.name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(folder), str(target))
            moved += 1

discarded = docs.loc[~docs["keep"], ["ticha_id", "language", "trptn_status", "image_count", "discard_reason"]]
discarded.to_csv(REPORTS_DIR / "discarded_documents.csv", index=False)
print(f"Kept {docs['keep'].sum()} documents ({int(docs.loc[docs['keep'], 'image_count'].sum())} pages, "
      f"{int(docs.loc[docs['keep'], 'transcribed_lines'].sum())} transcribed lines)")
print(f"Discarded {len(discarded)} documents; moved {moved} image folders to {EXCLUDED_DIR / 'images'}")
discarded["discard_reason"].value_counts().to_frame()

## 4. Segment the pages and pair lines with the transcription

1. **Segmentation**: kraken's default baseline model finds the lines of every page (kraken 7.1.1, fixed in `environment.yml`, so the same model is used everywhere). Folio numbers and words inserted between lines are kept as fragments without text.
2. **First pass** (only when fine-tuning here): a generic HTR model, McCATMuS, reads each line, and the transcription is cut into the word spans that match the readings. Confident pages go to `paired_mccatmus/auto/`.
3. **Fine-tuned model**: McCATMuS fine-tuned on those `auto/` lines reads Ticha's hands, Zapotec included, much better. `FINETUNED_SOURCE = "download"` takes the published model, which reproduces the published dataset; `"train"` fine-tunes it here (`FINETUNE_EPOCHS` epochs, batches of 16, on the CPU).
4. **Final pairing** with the fine-tuned model and the confidence settings of the config. Each pair is confident when its text fits the reading clearly better than the neighbouring lines' texts. Pages with at most `REVIEW_TOLERANCE` doubtful pairs go to `paired/auto/` (doubtful lines without text), the rest to `paired/review/`, and pages with no transcription to `paired/untranscribed/`.

Transcriptions are cleaned before pairing: Word markup pasted into some of them and line references such as `63r-5` are removed. Segmentation and readings are cached in `paired/segmentation/` and `paired/recognition/` (one cache per model), so re-running only processes what is missing. Line crops are cut in step 8, after review and splitting.


In [ ]:
def make_pairer(model) -> LinePairer:
    return LinePairer(
        segmenter=KrakenSegmenter(cache_dir=PAIRED_DIR / "segmentation", device=DEVICE),
        recognizer=KrakenRecognizer(model, cache_dir=PAIRED_DIR / "recognition", device=DEVICE),
        statuses=KEEP_STATUSES,
        cer_threshold=CER_THRESHOLD,
        max_cer=MAX_CER,
        min_margin=MIN_MARGIN,
        review_tolerance=REVIEW_TOLERANCE,
        export_lines=False,
    )


def finetuned_model() -> str:
    """Path of the model for the final pairing: published, trained here, or McCATMuS."""
    path = FINETUNED_DIR / FINETUNED_FILE
    if FINETUNED_SOURCE == "generic":
        return GENERIC_MODEL
    if path.exists():
        return str(path)
    if FINETUNED_SOURCE == "download":
        from huggingface_hub import hf_hub_download
        return hf_hub_download(FINETUNED_REPO, FINETUNED_FILE, local_dir=FINETUNED_DIR)

    # "train": first pass with McCATMuS, then fine-tune it on that pass's auto/ pages
    if not done(GENERIC_PAIRED_DIR / "manifest.csv"):
        make_pairer(GENERIC_MODEL).run(KEPT_TEXTS_CSV, KEPT_IMAGES_CSV, GENERIC_PAIRED_DIR, workers=WORKERS)
    ketos = str(Path(sys.executable).with_name("ketos"))
    base = str(KrakenRecognizer(GENERIC_MODEL).path)
    FINETUNED_DIR.mkdir(parents=True, exist_ok=True)
    arrow = FINETUNED_DIR / "auto.arrow"
    xmls = sorted(str(p) for p in (GENERIC_PAIRED_DIR / "auto").glob("*.xml"))
    threads = str(os.cpu_count() or 1)
    # Pre-extract the line images once (otherwise every epoch re-cuts them from the pages)
    subprocess.run([ketos, "-d", "cpu", "--workers", "4", "--threads", threads,
                    "compile", "-f", "page", "-o", str(arrow), *xmls], check=True)
    subprocess.run([ketos, "-d", DEVICE, "--workers", "2", "--threads", threads,
                    "train", "-f", "binary", "-i", base, "--resize", "union", "-B", "16",
                    "-q", "fixed", "-N", str(FINETUNE_EPOCHS), "-o", str(FINETUNED_DIR), str(arrow)],
                   check=True)
    checkpoints = sorted(p for p in FINETUNED_DIR.glob("checkpoint_*.ckpt") if "abort" not in p.name)
    subprocess.run([ketos, "convert", "-o", str(path), str(checkpoints[-1])], check=True)
    return str(path)


HTR_MODEL = finetuned_model()
print(f"Pairing model: {HTR_MODEL}")

manifest_csv = PAIRED_DIR / "manifest.csv"
if done(manifest_csv):
    manifest = pd.read_csv(manifest_csv)
    print(f"Loaded {manifest_csv}")
else:
    manifest = make_pairer(HTR_MODEL).run(KEPT_TEXTS_CSV, KEPT_IMAGES_CSV, PAIRED_DIR, workers=WORKERS)

paired_lines = pd.read_csv(PAIRED_DIR / "lines.csv")
manifest["status"].value_counts().to_frame("pages / documents")


## 5. Report: automatically paired vs. pages that need manual review

In [ ]:
pages = manifest[manifest["status"].isin(["auto", "review", "untranscribed", "error"])].copy()
pages["page"] = pages["page"].astype(int)
by_status = pages["status"].value_counts()
n_pages = len(pages)

doc_status = pages.groupby("doc_id")["status"].agg(lambda s: "all auto" if (s == "auto").all() else "needs review")
print(f"Pages: {n_pages}")
for status in ("auto", "review", "untranscribed", "error"):
    print(f"  {status:<7} {by_status.get(status, 0):>5}  ({100 * by_status.get(status, 0) / max(n_pages, 1):.1f}%)")
print(f"Documents: {len(doc_status)} -> {(doc_status == 'all auto').sum()} fully auto, "
      f"{(doc_status == 'needs review').sum()} with at least one page to review")
skipped = manifest[manifest["status"] == "skipped"]
if len(skipped):
    print(f"Documents skipped by the pairer: {len(skipped)}")
    display(skipped[["doc_id", "reason"]])

lines_ok = paired_lines[paired_lines["text"].notna()]
print(f"\nSegmented lines: {len(paired_lines)}; paired with text: {len(lines_ok)}; "
      f"confident: {int(paired_lines['confident'].sum())} "
      f"({100 * paired_lines['confident'].sum() / max(len(lines_ok), 1):.1f}% of paired)")

In [ ]:
# Why pages were sent to review (a page can have several reasons)
review = pages[pages["status"] == "review"].copy()
reasons = (
    review["reason"].fillna("").str.split("; ").explode()
    .str.replace(r"^\d+ ", "N ", regex=True)              # "3 low-confidence pair(s)" -> "N low-confidence pair(s)"
    .str.replace(r": \d+ segmented, \d+ transcribed", "", regex=True)
    .str.replace(r"without image: .*", "without image", regex=True)
)
reasons[reasons != ""].value_counts().to_frame("pages")

In [ ]:
# Which documents / pages require manual revision
review_docs = (
    review.groupby("doc_id")
    .agg(pages_to_review=("page", "size"),
         pages=("page", lambda s: ", ".join(map(str, s))),
         paired=("paired_lines", "sum"),
         confident=("confident_lines", "sum"))
)
review_docs["total_pages"] = pages.groupby("doc_id").size().reindex(review_docs.index)
review_docs["low_conf_lines"] = (review_docs["paired"] - review_docs["confident"]).astype(int)
review_docs = review_docs[["pages_to_review", "total_pages", "pages", "low_conf_lines"]].sort_values("pages_to_review", ascending=False)

review[["doc_id", "page", "folio", "reason", "segmented_lines", "paired_lines", "confident_lines",
        "unpaired_transcription_lines", "xml"]].to_csv(REPORTS_DIR / "review_pages.csv", index=False)
print(f"Full list of pages to review: {REPORTS_DIR / 'review_pages.csv'}")
review_docs

In [ ]:
errors = pages[pages["status"] == "error"]
errors[["doc_id", "page", "image", "reason"]] if len(errors) else print("No segmentation errors")

## 6. Review the pages in `review/`

Pages in `data/paired/review/` were not paired confidently enough to use as they are. Two ways to review them:

- **Decisions file (default, `REVIEW_DECISIONS`)**: `review/decisions.txt` in the repository records, for each line of those pages, whether its text was kept, fixed (with the corrected text) or dropped, after checking it against the image. The next cell applies it and writes the reviewed PAGE XML to `data/reviewed/`. Lines not listed keep their text only if the pair is confident. This reproduces the review of the published dataset, as long as segmentation and pairing are the same (same kraken version and pairing model). `ticha check -d data/paired -s review` draws the lines with their text, to review new pages the same way and add them to the file.
- **By hand in [eScriptorium](https://escriptorium.inria.fr)**: import the images and PAGE XML from `review/` (the `.txt` next to each page has the transcription assigned to it), fix the lines, export as PAGE XML and put the files in `data/reviewed/` with the same names. Pages already in `reviewed/` are not overwritten by the decisions file unless `OVERWRITE` is set.

A page in `review/` is used when its reviewed XML is in `reviewed/`; otherwise it is left out (or, with `INCLUDE_UNREVIEWED = True`, used with only its confident lines). Steps 7-10 can be re-run after more pages are reviewed: the split of the documents stays stable as long as the same documents are present.


In [ ]:
if REVIEW_DECISIONS and Path(REVIEW_DECISIONS).exists():
    applied = apply_review(PAIRED_DIR, REVIEW_DECISIONS, REVIEWED_DIR, min_conf=MIN_CONF,
                           skip_existing=not OVERWRITE)
    done_pages = applied[~applied["skipped"]]
    print(f"Applied {REVIEW_DECISIONS.name} to {len(done_pages)} review pages -> {REVIEWED_DIR} "
          f"({(applied['skipped']).sum()} already reviewed, left as they are)")
    print(f"Lines kept: {done_pages['kept'].sum()}, fixed: {done_pages['fixed'].sum()}, "
          f"left without text: {done_pages['dropped'].sum()}")
    if applied.attrs["unused"]:
        print(f"{len(applied.attrs['unused'])} decisions are for pages not in review in this run "
              "(different segmentation or pairing model?)")
else:
    print("No decisions file: put the PAGE XML reviewed by hand in", REVIEWED_DIR)


In [ ]:
final_pages = build.final_pages(DATA, INCLUDE_UNREVIEWED, MIN_CONF)
final_pages.to_csv(REPORTS_DIR / "final_pages.csv", index=False)
pending = final_pages[final_pages["source"] == "pending review"]
final_pages = final_pages[(final_pages["source"] != "pending review") & (final_pages["usable_lines"] > 0)]

print(f"Pages used: {len(final_pages)} ({int(final_pages['usable_lines'].sum())} lines)")
display(final_pages["source"].value_counts().to_frame("pages"))
print(f"Still pending review: {len(pending)} pages from {pending['doc_id'].nunique()} documents")
pending.groupby("doc_id")["page"].apply(lambda s: ", ".join(map(str, s))).to_frame("pages") if len(pending) else None

## 7. Split documents into train / validation / test (80 / 10 / 10)

The split is made by **document** (no document is shared between subsets, otherwise the same hand and text would leak into validation/test). Within each language, documents are shuffled with `SEED` and each one goes to the subset that is least filled relative to its target share of that language's **lines**. So every subset gets documents of every language (when a language has at least 3 documents) and the line shares end up close to 80/10/10.

The documents in `EXCLUDED_DOCS` ([`scripts/dataset_exclusions.py`](../scripts/dataset_exclusions.py)) are removed **after** the split, so every other document keeps its subset. Ticha catalogues some manuscripts twice; when the copies fall into different subsets, the one in test is left out.

In [ ]:
doc_table = build.split_documents(final_pages, SEED)
# Excluded after the split, so every other document keeps the subset it had before.
excluded = [doc_id for doc_id in EXCLUDED_DOCS if doc_id in doc_table.index]
doc_table = doc_table.drop(index=excluded)
final_pages = final_pages[~final_pages["doc_id"].isin(excluded)]
for doc_id in excluded:
    print(f"Excluded {doc_id}: {EXCLUDED_DOCS[doc_id]}")
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
doc_table.to_csv(SPLITS_DIR / "splits.csv")

split_summary = doc_table.groupby("split")[["pages", "lines"]].sum().reindex(SPLIT_RATIOS, fill_value=0)
split_summary.insert(0, "documents", doc_table["split"].value_counts().reindex(SPLIT_RATIOS, fill_value=0))
split_summary["% lines"] = (100 * split_summary["lines"] / split_summary["lines"].sum()).round(1)
split_summary["target %"] = [100 * r for r in SPLIT_RATIOS.values()]
display(split_summary)
print("Lines per language and subset")
doc_table.pivot_table(index="language", columns="split", values="lines", aggfunc="sum", fill_value=0).reindex(columns=list(SPLIT_RATIOS), fill_value=0)

## 8. Cut the lines and pair them with their transcription

For every page of each subset:

- `splits/<split>/auto/<doc>_page_NNN.{png,xml}`: the final page (image + PAGE XML with only the usable text). The folder is named `auto/` because the augmenter reads pages from there.
- `splits/<split>/lines/<doc>_page_NNN/line_NNN.png` + `line_NNN.gt.txt`: each usable line, cropped along its polygon (outside painted white), with its text. This is the format `ketos train -f path` reads.

`splits/<split>/` is rebuilt from scratch each time this cell runs (everything in it is derived from `paired/` and `reviewed/`).

In [ ]:
dataset_lines = build.cut_lines(final_pages, doc_table, SPLITS_DIR, MIN_CONF)
for split in SPLIT_RATIOS:
    build.write_list(build.line_images(SPLITS_DIR / split / "lines"), SPLITS_DIR / f"{split}.txt")
dataset_lines.groupby("split").size().reindex(SPLIT_RATIOS, fill_value=0).to_frame("line images")

## 9. Data augmentation (separate from the original dataset)

Augmented copies are written to **`data/augmented/`**, never into `splits/`, so the original dataset stays clean and can be used on its own:

- `augmented/train/lines/<page>/line_NNN_augKK.png` + `.gt.txt`: `COPIES` copies of each train line. `degrade` copies (`DEGRADE_RATIO`) imitate Ticha's untranscribed scans: lower resolution, greyish paper with a cool cast, faded brown ink, washed out or underexposed, uneven light, stains. `variety` copies add brightness/contrast/gamma, ink weight, blur, noise, JPEG, small rotation/slant/scale and elastic distortion.
- `augmented/test_degraded/lines/...`: `TEST_DEGRADED_COPIES` `degrade`-only copies of each test line (no `variety`, no geometry). Evaluate the model on `splits/test.txt` and on `augmented/test_degraded.txt` to see how much worse it reads low-quality documents.
- `augmented/validation_degraded/lines/...`: the same for validation (`VALIDATION_DEGRADED_COPIES`), to choose settings on low-quality lines without looking at test. It is not part of the published dataset.

Each set has its own fixed seed, so adding or removing one never changes the images of another.

File lists: `splits/{train,validation,test}.txt` (original lines) and `augmented/{train,test_degraded,validation_degraded}.txt`. The parameters of every copy are in `augmented/<name>/augment.csv`. The same steps 6-10 can be run without the notebook: `python scripts/build_dataset.py`.


In [ ]:
shutil.rmtree(AUGMENTED_DIR, ignore_errors=True)
# (subset, set name, copies, share of degrade copies, geometry, seed offset)
jobs = [("train", "train", COPIES, DEGRADE_RATIO, True, 0)]
if TEST_DEGRADED_COPIES:
    jobs.append(("test", "test_degraded", TEST_DEGRADED_COPIES, 1.0, False, 1))
if VALIDATION_DEGRADED_COPIES:
    jobs.append(("validation", "validation_degraded", VALIDATION_DEGRADED_COPIES, 1.0, False, 2))
aug_counts = {job[1]: build.augment_job(SPLITS_DIR, AUGMENTED_DIR, SEED, MIN_CONF, job) for job in jobs}

## 10. Final dataset metrics

In [ ]:
dataset_lines["chars"] = dataset_lines["text"].str.len()

metrics = pd.DataFrame({
    "documents": dataset_lines.groupby("split")["doc_id"].nunique(),
    "pages": dataset_lines.groupby("split")[["doc_id", "page"]].apply(lambda g: len(g.drop_duplicates())),
    "original lines": dataset_lines.groupby("split").size(),
    "augmented lines": pd.Series({"train": aug_counts.get("train", 0)}),  # degraded sets are reported apart
    "characters": dataset_lines.groupby("split")["chars"].sum(),
    "mean chars / line": dataset_lines.groupby("split")["chars"].mean().round(1),
    "distinct characters": dataset_lines.groupby("split")["text"].apply(lambda s: len(set("".join(s)))),
    "lines from reviewed pages": dataset_lines[dataset_lines["source"] == "reviewed"].groupby("split").size(),
}).reindex(SPLIT_RATIOS).fillna(0)
metrics["total lines"] = metrics["original lines"] + metrics["augmented lines"]
metrics["% original lines"] = (100 * metrics["original lines"] / metrics["original lines"].sum()).round(1)
metrics.loc["TOTAL"] = metrics.sum(numeric_only=True)
for col in ("mean chars / line", "distinct characters"):
    metrics.loc["TOTAL", col] = (round(dataset_lines["chars"].mean(), 1) if col.startswith("mean")
                                 else len(set("".join(dataset_lines["text"]))))
metrics = metrics.astype({c: int for c in metrics.columns if c not in ("mean chars / line", "% original lines")})
metrics.to_csv(REPORTS_DIR / "final_metrics.csv")
metrics

In [ ]:
print("Original lines per language and subset")
display(dataset_lines.pivot_table(index="language", columns="split", values="line", aggfunc="size", fill_value=0)
        .reindex(columns=list(SPLIT_RATIOS), fill_value=0))

# Characters the model will see in validation/test but never in training
train_chars = set("".join(dataset_lines.loc[dataset_lines["split"] == "train", "text"]))
for split in [s for s in SPLIT_RATIOS if s != "train"]:
    unseen = set("".join(dataset_lines.loc[dataset_lines["split"] == split, "text"])) - train_chars
    print(f"Characters in {split} missing from train: {''.join(sorted(unseen)) or '(none)'}")

# Line image sizes (heights matter for kraken's input normalisation)
sizes = []
for path in dataset_lines["image"]:
    with Image.open(path) as im:
        sizes.append(im.size)
dataset_lines["width"], dataset_lines["height"] = zip(*sizes) if sizes else ([], [])
print("\nLine image size (px)")
display(dataset_lines.groupby("split")[["width", "height"]].describe().round(0))

print("Most frequent characters")
pd.Series(list("".join(dataset_lines["text"]))).value_counts().head(40).to_frame("count").T

In [ ]:
summary = {
    "documents": int(dataset_lines["doc_id"].nunique()),
    "pages": int(len(final_pages)),
    "pages pending review": int(len(pending)),
    "original lines": int(len(dataset_lines)),
    "augmented lines": aug_counts,
    "dataset version": DATASET_VERSION,
    "excluded documents": {doc_id: EXCLUDED_DOCS[doc_id] for doc_id in excluded},
    "splits": {s: {"documents": int(metrics.loc[s, "documents"]), "lines": int(metrics.loc[s, "total lines"])}
               for s in SPLIT_RATIOS},
    "config": {"seed": SEED, "split_ratios": SPLIT_RATIOS, "keep_statuses": list(KEEP_STATUSES or []),
               "cer_threshold": CER_THRESHOLD, "min_conf": MIN_CONF, "include_unreviewed": INCLUDE_UNREVIEWED,
               "copies": COPIES, "degrade_ratio": DEGRADE_RATIO, "test_degraded_copies": TEST_DEGRADED_COPIES,
               "validation_degraded_copies": VALIDATION_DEGRADED_COPIES, "htr_model": HTR_MODEL},
}
(REPORTS_DIR / "final_metrics.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nTrain with kraken, e.g. (original + augmented train lines, clean validation):")
print(f"  cat {SPLITS_DIR / 'train.txt'} {AUGMENTED_DIR / 'train.txt'} > train_all.txt")
print(f"  ketos train -f path -t train_all.txt -e {SPLITS_DIR / 'validation.txt'}")
print(f"Test on real scans and on degraded copies:")
print(f"  ketos test -f path -m model.safetensors -e {SPLITS_DIR / 'test.txt'}")
print(f"  ketos test -f path -m model.safetensors -e {AUGMENTED_DIR / 'test_degraded.txt'}")

## Next steps

- **Publish the dataset**: `python scripts/export_hf_dataset.py` turns `splits/` and `augmented/` into the Parquet folder published as [`maclenn77/ticha-htr-lines`](https://huggingface.co/datasets/maclenn77/ticha-htr-lines) (the `validation_degraded` set is not exported).
- **Train TrOCR** on the published dataset: [`train/ticha_trocr_finetuning.ipynb`](train/ticha_trocr_finetuning.ipynb), then the right-to-left reranker in [`train/ticha_trocr_beams_r2l_rerank.ipynb`](train/ticha_trocr_beams_r2l_rerank.ipynb).
- **Read new lines** with the trained models: [`ticha_trocr_usage_example.ipynb`](ticha_trocr_usage_example.ipynb).